In [0]:
%pip install openpyxl "pymongo[srv]" certifi

In [0]:
dbutils.library.restartPython()

In [0]:
%run ./00_setup_mongodb

In [0]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import certifi

mongo_uri = dbutils.secrets.get(
    scope="mongodb",
    key="uri"
)

client = MongoClient(
    mongo_uri,
    server_api=ServerApi("1"),
    tls=True,
    tlsCAFile=certifi.where(),
    serverSelectionTimeoutMS=60000
)

client.admin.command("ping")

print("Conexão com MongoDB Atlas realizada com sucesso!")

In [0]:
mongo_database = "databricks_learning"

collection_origem = "chamados_raw"
collection_destino = "chamados_insights"

db = client[mongo_database]

chamados_raw = db[collection_origem]
chamados_insights = db[collection_destino]

print("Origem:", collection_origem)
print("Destino:", collection_destino)

In [0]:
documentos_raw = list(chamados_raw.find({}))

print("Quantidade de chamados lidos:", len(documentos_raw))

if not documentos_raw:
    raise ValueError("Nenhum chamado encontrado na collection chamados_raw.")

In [0]:
df_chamados = pd.DataFrame(documentos_raw)

print("DataFrame criado com sucesso!")
print("Linhas:", len(df_chamados))
print("Colunas:", df_chamados.columns.tolist())

display(df_chamados.head(10))

In [0]:
def chamado_esta_aberto(status):
    return status in ["Aberto", "Em atendimento", "Pendente"]


def calcular_horas(inicio, fim):
    if inicio is None or pd.isna(inicio):
        return None

    if fim is None or pd.isna(fim):
        fim = datetime.now()

    diferenca = fim - inicio

    return round(diferenca.total_seconds() / 3600, 2)


def classificar_risco_sla(status, tempo_horas, sla_horas):
    if tempo_horas is None or sla_horas is None:
        return "Indefinido"

    if chamado_esta_aberto(status):
        if tempo_horas > sla_horas:
            return "Alto"
        elif tempo_horas >= sla_horas * 0.7:
            return "Médio"
        else:
            return "Baixo"

    if tempo_horas > sla_horas:
        return "Encerrado fora do SLA"

    return "Encerrado dentro do SLA"


def recomendar_acao(status, risco_sla):
    if chamado_esta_aberto(status) and risco_sla == "Alto":
        return "Priorizar atendimento imediatamente"

    if chamado_esta_aberto(status) and risco_sla == "Médio":
        return "Monitorar chamado e acelerar tratativa"

    if chamado_esta_aberto(status) and risco_sla == "Baixo":
        return "Manter acompanhamento normal"

    if risco_sla == "Encerrado fora do SLA":
        return "Analisar causa do atraso e registrar melhoria"

    if risco_sla == "Encerrado dentro do SLA":
        return "Nenhuma ação necessária"

    return "Revisar dados do chamado"

In [0]:
def gerar_insight_chamado(chamado):
    status = chamado.get("status")
    data_abertura = chamado.get("data_abertura")
    data_fechamento = chamado.get("data_fechamento")
    sla_horas = chamado.get("sla_horas")

    if chamado_esta_aberto(status):
        tempo_horas = calcular_horas(data_abertura, None)
        tipo_tempo = "tempo_aberto_horas"
    else:
        tempo_horas = calcular_horas(data_abertura, data_fechamento)
        tipo_tempo = "tempo_resolucao_horas"

    risco_sla = classificar_risco_sla(
        status=status,
        tempo_horas=tempo_horas,
        sla_horas=sla_horas
    )

    acao_recomendada = recomendar_acao(
        status=status,
        risco_sla=risco_sla
    )

    cliente = chamado.get("cliente", {})

    insight = {
        "_id": chamado.get("_id"),
        "id_chamado": chamado.get("id_chamado"),
        "cliente": {
            "nome": cliente.get("nome"),
            "segmento": cliente.get("segmento"),
            "cidade": cliente.get("cidade"),
            "uf": cliente.get("uf")
        },
        "categoria": chamado.get("categoria"),
        "prioridade": chamado.get("prioridade"),
        "status": status,
        "sla_horas": sla_horas,
        tipo_tempo: tempo_horas,
        "risco_sla": risco_sla,
        "acao_recomendada": acao_recomendada,
        "qtd_interacoes": chamado.get("qtd_interacoes"),
        "satisfacao": chamado.get("satisfacao"),
        "data_abertura": data_abertura,
        "data_fechamento": data_fechamento,
        "origem": {
            "collection": collection_origem,
            "processado_por": "databricks",
            "data_processamento": datetime.now()
        }
    }

    return insight

In [0]:
insights = [
    gerar_insight_chamado(chamado)
    for chamado in documentos_raw
]

print("Quantidade de insights gerados:", len(insights))

print("Exemplo de insight:")
print(json.dumps(insights[0], indent=2, ensure_ascii=False, default=str))

In [0]:
operacoes = [
    ReplaceOne(
        {"_id": insight["_id"]},
        insight,
        upsert=True
    )
    for insight in insights
]

resultado = chamados_insights.bulk_write(operacoes)

print("Insights enviados com sucesso!")
print("Matched:", resultado.matched_count)
print("Modified:", resultado.modified_count)
print("Upserted:", resultado.upserted_count)

In [0]:
total_insights = chamados_insights.count_documents({})

print("Total de documentos em chamados_insights:", total_insights)

exemplo_insight = chamados_insights.find_one()

print("Exemplo salvo:")
print(json.dumps(exemplo_insight, indent=2, ensure_ascii=False, default=str))

In [0]:
pipeline_risco = [
    {
        "$group": {
            "_id": "$risco_sla",
            "quantidade": {"$sum": 1}
        }
    },
    {
        "$sort": {
            "quantidade": -1
        }
    }
]

for item in chamados_insights.aggregate(pipeline_risco):
    print(item)